# 평가 데이터 생성 및 비교 전체 코드
- Vector DB를 불러와서 Base 모델과 RAG 모델의 응답을 비교하고 CSV로 저장하는 전체 코드

In [6]:
import os
import json
import pandas as pd
import getpass
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
from tqdm import tqdm  # 진행률 표시를 위한 라이브러리

# ==========================================
# 1. 초기 설정 및 보안 인증
# ==========================================
MY_API_KEY = getpass.getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)

# 경로 설정 (evaluation 폴더 기준 상위 폴더의 data 참조)
CHROMA_PATH = "../data/test/card_vector_db"

# Embedding 함수 설정 (기존 인덱싱 때와 동일해야 함)
ko_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

# Vector DB 연결 (기존 컬렉션 불러오기)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(
    name="card_separated_collection",
    embedding_function=ko_embedding_func
)

# ==========================================
# 2. 모델 응답 함수 정의
# ==========================================

# (1) Base GPT-3.5-turbo 응답 함수 (RAG 미적용)
def get_base_model_response(query):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "당신은 유능한 금융 상품 추천 전문가입니다. 아는 지식 내에서 답변해주세요."},
            {"role": "user", "content": query}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

# (2) RAG 구성 후 응답 함수 (기존 로직 유지)
def get_rag_model_response(query, persona_dict, target_card_type=None):
    persona_traits = persona_dict['traits']
    search_query = f"{persona_traits} {query}"
    
    # DB 검색
    if target_card_type in ["체크", "신용"]:
        results = collection.query(
            query_texts=[search_query], 
            n_results=15, # 평가 효율을 위해 결과 수 조정 가능
            where={"card_type": target_card_type}
        )
    else:
        results = collection.query(query_texts=[search_query], n_results=15)

    raw_documents = results['documents'][0]
    available_card_names = set()
    card_context_map = {}
    
    for doc in raw_documents:
        first_line = doc.split("\n")[0]
        card_title = first_line.split("카드명: ")[1].split("|")[0].strip()
        available_card_names.add(card_title)
        if card_title not in card_context_map: card_context_map[card_title] = []
        card_context_map[card_title].append(doc)

    verified_list_str = ", ".join(list(available_card_names))
    formatted_context = ""
    for title, contents in card_context_map.items():
        formatted_context += f"\n### 카드 데이터: {title} ###\n" + "\n".join(contents) + "\n"

    system_prompt = f"""
    당신은 금융 상품 추천 전문가입니다. 반드시 [검증된 카드 리스트] 내의 카드만 추천하십시오.
    [검증된 카드 리스트]: {verified_list_str}
    [상세 데이터]: {formatted_context}
    [사용자 페르소나]: {persona_dict['name']} ({persona_traits})
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

# ==========================================
# 3. 테스트 시나리오 설정
# ==========================================
test_scenarios = [
    {"persona": {"name": "20대 자기계발생", "traits": "무실적 카드 선호, 대중교통 및 학원 이용 잦음"}, "query": "전월 실적 부담 없으면서 대중교통, 학원 혜택이 좋은 체크카드 알려줘.", "target": "체크"},
    {"persona": {"name": "20대 사회초년생 자취형", "traits": "배달·카페·온라인 쇼핑 비중 높음"}, "query": "배달 음식과 카페, 온라인 쇼핑에서 자주 쓰기 좋은 체크카드 추천해줘.", "target": "체크"},
    {"persona": {"name": "20대 외출 활동형", "traits": "외식과 카페 이용 빈도 높음, 문화생활"}, "query": "카페와 외식 위주로 자주 쓰기 좋은 체크카드를 알려줘.", "target": "체크"},
    {"persona": {"name": "30대 출퇴근 직장인", "traits": "대중교통 출퇴근, 점심 외식과 커피 지출"}, "query": "출퇴근 교통비랑 회사 근처 카페 혜택이 집중된 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 자차 보유 직장인", "traits": "자차 출퇴근, 주유비 지출 큼"}, "query": "주유 혜택이 크고 자차 이용자에게 도움이 되는 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 여행 중심형", "traits": "해외 여행 및 직구 빈번, 항공권·숙박"}, "query": "해외 결제 수수료 혜택이 좋고 여행 시 활용하기 좋은 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 건강관리 소비형", "traits": "병원, 약국, 헬스장 이용 빈도 높음"}, "query": "병원이나 헬스장 지출에 혜택을 받을 수 있는 신용카드를 알려줘.", "target": "신용"},
    {"persona": {"name": "40대 가족 생활형", "traits": "대형마트 장보기, 생활비와 고정비 관리"}, "query": "대형마트 장보기랑 생활비 지출에 혜택이 좋은 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "40대 교육비 집중형", "traits": "자녀 학원비 결제 비중 큼"}, "query": "자녀 학원비 결제 시 혜택을 받을 수 있는 신용카드를 알려줘.", "target": "신용"},
    {"persona": {"name": "디지털 구독 소비형", "traits": "OTT, 온라인 쇼핑, 구독 서비스"}, "query": "온라인 결제나 구독 서비스에서 할인/적립이 있는 체크카드 추천해줘.", "target": "체크"}
]

# ==========================================
# 4. 평가 실행 및 CSV 저장
# ==========================================
ITERATIONS = 10 # 페르소나별 10회 반복 (총 100회)
evaluation_data = []

print(f"🚀 테스트 시작 (총 {len(test_scenarios) * ITERATIONS}건)")

for i, sc in enumerate(test_scenarios):
    print(f"📍 [{i+1}/10] {sc['persona']['name']} 시나리오 진행 중...")
    
    for run in tqdm(range(1, ITERATIONS + 1), desc="반복 회차"):
        # 응답 생성
        base_ans = get_base_model_response(sc['query'])
        rag_ans = get_rag_model_response(sc['query'], sc['persona'], target_card_type=sc['target'])
        
        evaluation_data.append({
            "시나리오_ID": i + 1,
            "회차": run,
            "페르소나": sc['persona']['name'],
            "요청_카드유형": sc['target'],
            "질문": sc['query'],
            "Base_GPT_응답": base_ans,
            "RAG_시스템_응답": rag_ans,
            "Human Evaluation": "",    # 직접 기입용
            "만족도(상/중/하)": "" # 직접 기입용
        })

# 데이터프레임 저장
df = pd.DataFrame(evaluation_data)
output_file = "model_evaluation.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\n✅ 평가 데이터 생성이 완료되었습니다: {output_file}")

🚀 테스트 시작 (총 100건)
📍 [1/10] 20대 자기계발생 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [00:54<00:00,  5.49s/it]


📍 [2/10] 20대 사회초년생 자취형 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [01:05<00:00,  6.57s/it]


📍 [3/10] 20대 외출 활동형 시나리오 진행 중...


...

📍 [9/10] 40대 교육비 집중형 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [01:45<00:00, 10.52s/it]


📍 [10/10] 디지털 구독 소비형 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [01:47<00:00, 10.71s/it]


✅ 평가 데이터 생성이 완료되었습니다: model_evaluation.csv
